# Module 1 — Data Pipeline

This notebook scrapes 69 books from three Books to Scrape categories, cleans and converts the data, stores it in normalized SQLite tables, runs five SQL queries, and verifies the SQL JOIN with `pandas.merge`.

**Fixed assignment conversion rate: 1 GBP = 105.50 INR.** This is a project-defined baseline, not a live exchange rate.

In [1]:
!pip -q install beautifulsoup4 requests

In [2]:
import sqlite3
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = 'https://books.toscrape.com/'
TARGET_CATEGORIES = ('Travel', 'Mystery', 'Historical Fiction')
GBP_TO_INR = 105.50
RATING_MAP = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
DATABASE = Path('books.db')

def fetch(url):
    response = requests.get(url, timeout=30, headers={'User-Agent': 'colab-data-pipeline/1.0'})
    response.raise_for_status()  # explicit HTTP error handling
    return BeautifulSoup(response.text, 'html.parser')

def scrape_books():
    home = fetch(BASE_URL)
    category_urls = {a.get_text(strip=True): requests.compat.urljoin(BASE_URL, a['href'])
                     for a in home.select('ul.nav-list ul a')}
    missing = set(TARGET_CATEGORIES) - set(category_urls)
    if missing:
        raise ValueError(f'Missing source categories: {sorted(missing)}')

    rows = []
    for category in TARGET_CATEGORIES:
        url = category_urls[category]
        while url:
            soup = fetch(url)
            for card in soup.select('article.product_pod'):
                classes = card.select_one('p.star-rating').get('class', [])
                text_rating = next((x for x in classes if x in RATING_MAP), '')
                rows.append({
                    'title': card.select_one('h3 a')['title'].strip(),
                    'price': card.select_one('p.price_color').get_text(strip=True),
                    'star_rating': text_rating,
                    'availability': card.select_one('p.instock.availability').get_text(' ', strip=True),
                    'category': category
                })
            next_link = soup.select_one('li.next a')
            url = requests.compat.urljoin(url, next_link['href']) if next_link else ''
    return pd.DataFrame(rows)


In [3]:
# Scrape raw fields required by the assignment
raw = scrape_books()
assert len(raw) >= 60 and raw['category'].nunique() >= 3
print(f'Scraped {len(raw)} books across {raw.category.nunique()} categories.')
display(raw.head())
raw.to_csv('books_raw.csv', index=False)

Scraped 69 books across 3 categories.


,title,price,star_rating,availability,category
0,It's Only the Himalayas,Â£45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,Â£37.33,Three,In stock,Travel


## Cleaning choices

The currency symbol is removed from price, ratings are mapped from words to integers, and availability becomes a boolean. Malformed numeric values are median-imputed so a messy row does not crash the pipeline. Rows with no title/category are dropped because those relational fields cannot be responsibly inferred.

In [4]:
clean = raw.copy()
clean['price_gbp'] = pd.to_numeric(clean['price'].str.replace(r'[^0-9.]', '', regex=True), errors='coerce')
clean['rating'] = clean['star_rating'].map(RATING_MAP)
clean['in_stock'] = clean['availability'].str.contains('in stock', case=False, na=False)

for column in ['price_gbp', 'rating']:
    median = clean[column].median()
    if pd.isna(median):
        raise ValueError(f'Cannot impute {column}: no valid scraped values.')
    clean[column] = clean[column].fillna(median)

clean = clean.dropna(subset=['title', 'category'])
clean = clean[(clean.title.str.strip() != '') & (clean.category.str.strip() != '')]
clean['price_gbp'] = clean['price_gbp'].astype(float)
clean['rating'] = clean['rating'].round().astype(int)
clean['price_inr'] = (clean['price_gbp'] * GBP_TO_INR).round(2)
clean = clean[['title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category']].reset_index(drop=True)
clean.to_csv('books_clean.csv', index=False)
print(clean.dtypes)
display(clean.head())

title         object
price_gbp    float64
price_inr    float64
rating         int64
in_stock        bool
category      object
dtype: object


,title,price_gbp,price_inr,rating,in_stock,category
0,It's Only the Himalayas,45.17,4765.44,2,True,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.86,4,True,Travel
2,See America: A Celebration of Our National Par...,48.87,5155.78,3,True,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.17,2,True,Travel
4,Under the Tuscan Sun,37.33,3938.31,3,True,Travel


In [5]:
# Normalized SQLite schema: categories (PK) -> books (FK)
with sqlite3.connect(DATABASE) as conn:
    conn.execute('PRAGMA foreign_keys = ON')
    conn.executescript('''
        DROP TABLE IF EXISTS books;
        DROP TABLE IF EXISTS categories;
        CREATE TABLE categories (
          category_id INTEGER PRIMARY KEY,
          category_name TEXT NOT NULL UNIQUE
        );
        CREATE TABLE books (
          book_id INTEGER PRIMARY KEY,
          title TEXT NOT NULL, price_gbp REAL NOT NULL, price_inr REAL NOT NULL,
          rating INTEGER NOT NULL CHECK(rating BETWEEN 1 AND 5),
          in_stock INTEGER NOT NULL CHECK(in_stock IN (0,1)),
          category_id INTEGER NOT NULL REFERENCES categories(category_id)
        );
    ''')
    categories = pd.DataFrame({'category_name': sorted(clean.category.unique())})
    categories.to_sql('categories', conn, if_exists='append', index=False)
    ids = pd.read_sql_query('SELECT category_id, category_name FROM categories', conn)
    books = clean.merge(ids, left_on='category', right_on='category_name', how='left')
    books = books[['title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category_id']].copy()
    books['in_stock'] = books['in_stock'].astype(int)
    books.to_sql('books', conn, if_exists='append', index=False)

print('SQLite database created:', DATABASE)

SQLite database created: books.db


In [6]:
queries = {
 '1. WHERE + ORDER BY + LIMIT': '''SELECT title, rating, price_gbp FROM books WHERE in_stock=1 ORDER BY rating DESC, price_gbp DESC LIMIT 10''',
 '2. DISTINCT': '''SELECT DISTINCT rating FROM books ORDER BY rating DESC''',
 '3. BETWEEN': '''SELECT title, price_inr FROM books WHERE price_inr BETWEEN 1000 AND 3000 ORDER BY price_inr LIMIT 10''',
 '4. IN': '''SELECT title, rating FROM books WHERE rating IN (4,5) ORDER BY rating DESC, title LIMIT 10''',
 '5. JOIN': '''SELECT b.title, c.category_name, b.rating, b.price_inr FROM books b JOIN categories c ON b.category_id=c.category_id ORDER BY b.rating DESC, b.price_inr DESC LIMIT 10'''
}

with sqlite3.connect(DATABASE) as conn:
    for name, query in queries.items():
        print('\n' + '=' * 70 + '\n' + name + '\nSQL: ' + query)
        display(pd.read_sql_query(query, conn))


1. WHERE + ORDER BY + LIMIT
SQL: SELECT title, rating, price_gbp FROM books WHERE in_stock=1 ORDER BY rating DESC, price_gbp DESC LIMIT 10


,title,rating,price_gbp
0,A Flight of Arrows (The Pathfinders #2),5,55.53
1,The Bachelor Girl's Guide to Murder (Herringfo...,5,52.30
2,A Time of Torment (Charlie Parker #14),5,48.35
3,While You Were Mine,5,41.32
4,The Red Tent,5,35.66
5,Mrs. Houdini,5,30.25
6,The Passion of Dolssa,5,28.32
7,"1,000 Places to See Before You Die",5,26.08
8,What Happened on Beale Street (Secrets of the ...,5,25.37
9,The Silkworm (Cormoran Strike #2),5,23.05



2. DISTINCT
SQL: SELECT DISTINCT rating FROM books ORDER BY rating DESC


,rating
0,5
1,4
2,3
3,2
4,1



3. BETWEEN
SQL: SELECT title, price_inr FROM books WHERE price_inr BETWEEN 1000 AND 3000 ORDER BY price_inr LIMIT 10


,title,price_inr
0,Tastes Like Fear (DI Marnie Rome #3),1127.79
1,Hide Away (Eve Duncan #20),1249.12
2,The Girl You Lost,1296.59
3,Playing with Fire,1446.41
4,That Darkness (Gardiner and Renner #1),1468.56
5,The Girl In The Ice (DCI Erika Foster #1),1672.18
6,The Constant Princess (The Tudor Court #1),1753.41
7,A Murder in Time,1755.52
8,A Study in Scarlet (Sherlock Holmes #1),1765.02
9,A Spy's Devotion (The Regency Spies of London #1),1790.33



4. IN
SQL: SELECT title, rating FROM books WHERE rating IN (4,5) ORDER BY rating DESC, title LIMIT 10


,title,rating
0,"1,000 Places to See Before You Die",5
1,A Flight of Arrows (The Pathfinders #2),5
2,A Spy's Devotion (The Regency Spies of London #1),5
3,A Time of Torment (Charlie Parker #14),5
4,Between Shades of Gray,5
5,Mrs. Houdini,5
6,The Bachelor Girl's Guide to Murder (Herringfo...,5
7,The Girl You Lost,5
8,The Passion of Dolssa,5
9,The Red Tent,5



5. JOIN
SQL: SELECT b.title, c.category_name, b.rating, b.price_inr FROM books b JOIN categories c ON b.category_id=c.category_id ORDER BY b.rating DESC, b.price_inr DESC LIMIT 10


,title,category_name,rating,price_inr
0,A Flight of Arrows (The Pathfinders #2),Historical Fiction,5,5858.42
1,The Bachelor Girl's Guide to Murder (Herringfo...,Mystery,5,5517.65
2,A Time of Torment (Charlie Parker #14),Mystery,5,5100.92
3,While You Were Mine,Historical Fiction,5,4359.26
4,The Red Tent,Historical Fiction,5,3762.13
5,Mrs. Houdini,Historical Fiction,5,3191.38
6,The Passion of Dolssa,Historical Fiction,5,2987.76
7,"1,000 Places to See Before You Die",Travel,5,2751.44
8,What Happened on Beale Street (Secrets of the ...,Mystery,5,2676.54
9,The Silkworm (Cormoran Strike #2),Mystery,5,2431.78


In [7]:
# Read at least two SQL results into pandas, then reproduce the JOIN with pd.merge.
with sqlite3.connect(DATABASE) as conn:
    where_result = pd.read_sql_query(queries['1. WHERE + ORDER BY + LIMIT'], conn)
    sql_join = pd.read_sql_query(queries['5. JOIN'], conn)
    books_df = pd.read_sql_query('SELECT * FROM books', conn)
    categories_df = pd.read_sql_query('SELECT * FROM categories', conn)

pandas_merge = (books_df.merge(categories_df, on='category_id', how='inner')
                [['title', 'category_name', 'rating', 'price_inr']]
                .sort_values(['rating', 'price_inr'], ascending=[False, False])
                .head(10).reset_index(drop=True))
print('SQL JOIN and pandas merge equivalent:', sql_join.reset_index(drop=True).equals(pandas_merge))
print('\nSQL JOIN result:')
display(sql_join)
print('pandas merge result:')
display(pandas_merge)

# Optional: download all submission artefacts from Colab.
from google.colab import files
files.download(str(DATABASE))
files.download('books_raw.csv')
files.download('books_clean.csv')

SQL JOIN and pandas merge equivalent: True

SQL JOIN result:


,title,category_name,rating,price_inr
0,A Flight of Arrows (The Pathfinders #2),Historical Fiction,5,5858.42
1,The Bachelor Girl's Guide to Murder (Herringfo...,Mystery,5,5517.65
2,A Time of Torment (Charlie Parker #14),Mystery,5,5100.92
3,While You Were Mine,Historical Fiction,5,4359.26
4,The Red Tent,Historical Fiction,5,3762.13
5,Mrs. Houdini,Historical Fiction,5,3191.38
6,The Passion of Dolssa,Historical Fiction,5,2987.76
7,"1,000 Places to See Before You Die",Travel,5,2751.44
8,What Happened on Beale Street (Secrets of the ...,Mystery,5,2676.54
9,The Silkworm (Cormoran Strike #2),Mystery,5,2431.78


pandas merge result:


,title,category_name,rating,price_inr
0,A Flight of Arrows (The Pathfinders #2),Historical Fiction,5,5858.42
1,The Bachelor Girl's Guide to Murder (Herringfo...,Mystery,5,5517.65
2,A Time of Torment (Charlie Parker #14),Mystery,5,5100.92
3,While You Were Mine,Historical Fiction,5,4359.26
4,The Red Tent,Historical Fiction,5,3762.13
5,Mrs. Houdini,Historical Fiction,5,3191.38
6,The Passion of Dolssa,Historical Fiction,5,2987.76
7,"1,000 Places to See Before You Die",Travel,5,2751.44
8,What Happened on Beale Street (Secrets of the ...,Mystery,5,2676.54
9,The Silkworm (Cormoran Strike #2),Mystery,5,2431.78


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>